# Reconnaissance vocale (ASR) en éwé — Fine-tuning de Whisper

**ASR** = *Automatic Speech Recognition* : transformer un **son** (la voix) en **texte**.
Ici, on apprend au modèle **Whisper** d'OpenAI à transcrire de l'**éwé** parlé, à partir
du jeu de données `romaricnadjire/ewe-asr-whisper` (versets bibliques lus à voix haute).

## Comment marche un système ASR moderne ?

On ne donne pas le son « brut » au modèle. On le transforme d'abord en une image du son
appelée **spectrogramme de Mel** (l'énergie du signal par fréquence et par instant, sur une
échelle proche de l'oreille humaine). Whisper est un *encodeur-décodeur* :

- l'**encodeur** lit le spectrogramme et en extrait un résumé ;
- le **décodeur** génère le texte, un token à la fois, comme un modèle de langage.

```mermaid
flowchart LR
    A[Audio .flac 48 kHz] -->|reechantillonnage 16 kHz| B[Forme d onde]
    B -->|log-Mel spectrogramme| C[input_features 80 x T]
    C --> D[Encodeur Whisper]
    D --> E[Decodeur Whisper]
    E -->|tokens| F[Texte eve]
```

> **Note importante :** l'éwé ne fait pas partie des ~99 langues officiellement reconnues
> par Whisper. Ce n'est pas bloquant : le *spectrogramme* est indépendant de la langue, et
> le *tokenizer* de Whisper est « byte-level » — il sait donc encoder les caractères éwé
> (ɖ, ɔ, ŋ, ɛ…). On utilisera un token de langue « porteur » (anglais) et le fine-tuning
> apprendra au modèle à produire de l'éwé après ce token. Pour une alternative native éwé,
> voir la dernière section (modèle **MMS** de Meta).


## 0. Installation des dépendances

- `transformers`, `datasets` : modèle + données
- `evaluate`, `jiwer` : métrique **WER** (taux d'erreur sur les mots)
- `librosa`, `soundfile` : lecture/rééchantillonnage audio
- `accelerate` : entraînement optimisé

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate librosa soundfile

## 1. Imports et configuration

Comme pour la traduction, on force **un seul GPU** (Kaggle T4 x2) et on réduit la
fragmentation mémoire **avant** d'importer `torch`.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import torch
import evaluate
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Configuration ----
MODEL_NAME  = "openai/whisper-small"     # 244M params, tient sur un T4
OUTPUT_DIR  = "./output/whisper-ewe"
DATASET_ID  = "romaricnadjire/ewe-asr-whisper"
LANG_CARRIER = "english"                 # slot de langue 'porteur' (eve non supporte)
SAMPLING_RATE = 16_000                    # Whisper attend du 16 kHz
MAX_LABEL_LEN = 225                       # longueur max de la transcription (tokens)
MAX_TRAIN_SAMPLES = None                  # mettre p.ex. 4000 pour un essai rapide

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("device :", device)


## 2. Chargement du dataset audio

Le dataset suit le format **audiofolder** : chaque split a un `metadata.jsonl` qui associe
un fichier `.flac` à sa `transcription` en éwé.

Les fichiers sont en **48 kHz**, mais Whisper exige du **16 kHz**. La ligne
`cast_column("audio", Audio(sampling_rate=16_000))` programme un rééchantillonnage
**automatique et paresseux** : la conversion n'a lieu qu'au moment où on lit réellement
chaque fichier.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass  # local : huggingface-cli login ou variable d'environnement HF_TOKEN

ds = load_dataset(DATASET_ID, token=True)
ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

if MAX_TRAIN_SAMPLES:
    ds["train"] = ds["train"].select(range(min(MAX_TRAIN_SAMPLES, len(ds["train"]))))

print(ds)


In [ ]:
# Inspecter un exemple : forme d'onde + transcription + duree
ex = ds["train"][0]
print("Transcription :", ex["transcription"])
print("Sampling rate :", ex["audio"]["sampling_rate"])
print("Nb echantillons :", len(ex["audio"]["array"]))
print("Duree (s) :", round(len(ex["audio"]["array"]) / ex["audio"]["sampling_rate"], 2))


## 3. Le processeur Whisper

Le `WhisperProcessor` réunit **deux** outils :

1. le **`feature_extractor`** : transforme la forme d'onde en **spectrogramme log-Mel**
   (80 canaux de fréquence). C'est l'« image » du son que lit l'encodeur ;
2. le **`tokenizer`** : transforme le texte éwé en `input_ids` (et inversement). Il ajoute
   au début des tokens spéciaux : `<|startoftranscript|>`, un token de **langue**, puis
   `<|transcribe|>`.

Comme l'éwé n'existe pas dans Whisper, on fixe la langue « porteuse » `english`. Le
fine-tuning réécrira le comportement pour produire de l'éwé.

In [ ]:
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language=LANG_CARRIER, task="transcribe"
)

# id du token de fin de transcription (sert au data collator plus bas)
print("Tokens speciaux de prefixe :")
print(processor.tokenizer.convert_ids_to_tokens(
    processor.tokenizer.prefix_tokens
))


## 4. Préparation des données

`prepare_dataset` applique, sur chaque exemple, les deux transformations clés :

- `input_features` = spectrogramme log-Mel calculé par le `feature_extractor` ;
- `labels` = la transcription encodée en `input_ids` par le `tokenizer`.

On supprime les colonnes d'origine (`audio`, `transcription`) car le `Trainer` n'a besoin
que de `input_features` et `labels`. On filtre aussi les audios trop longs (> 30 s), que
l'encodeur de Whisper ne peut pas traiter.

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]
    # 1) son -> spectrogramme log-Mel
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    # duree utile pour filtrer ensuite
    batch["input_length"] = len(audio["array"]) / audio["sampling_rate"]
    # 2) texte -> tokens
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

ds_prep = ds.map(
    prepare_dataset,
    remove_columns=ds["train"].column_names,
    desc="Extraction des features audio",
)

# Garder uniquement les audios <= 30 s
ds_prep = ds_prep.filter(lambda l: l < 30.0, input_columns=["input_length"])
print(ds_prep)


## 5. Le *data collator* (padding dynamique)

Les spectrogrammes et les transcriptions ont des **longueurs variables**. Le collator
construit chaque batch en complétant (*padding*) :

- les `input_features` via le `feature_extractor` ;
- les `labels` via le `tokenizer`. Les positions de padding des labels sont mises à
  **-100** pour être **ignorées** par la fonction de perte.

On retire aussi le token de début s'il a déjà été ajouté (le modèle le rajoute lui-même).

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        # 1) padding des spectrogrammes
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        # 2) padding des labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        # 3) padding -> -100 (ignore par la cross-entropy)
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # 4) retirer le token BOS si deja present (le modele le rajoute)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


## 6. La métrique : WER (et CER)

Le **WER** (*Word Error Rate*) mesure le pourcentage de mots faux (insertions, suppressions,
substitutions) par rapport à la transcription de référence. **Plus c'est bas, mieux c'est.**
Le **CER** fait pareil au niveau des **caractères** — pertinent pour l'éwé (morphologie
riche, beaucoup de signes diacritiques).

In [ ]:
metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    # remettre le pad_token a la place des -100 pour pouvoir decoder
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}


## 7. Chargement du modèle

On charge Whisper, puis on désactive les contraintes de génération héritées (`forced_decoder_ids`,
`suppress_tokens`) pour laisser le fine-tuning s'exprimer. `use_cache=False` est requis
lorsqu'on active le *gradient checkpointing* (économie de VRAM).

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

# Laisser le modele apprendre librement (pas de tokens imposes/suppimes)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

# A la generation (eval), reutiliser le prefixe 'porteur' english/transcribe
model.generation_config.language = LANG_CARRIER
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None


## 8. Entraînement

Réglages adaptés à un T4. `predict_with_generate=True` est **indispensable** : pour calculer
le WER, il faut que le modèle **génère** réellement les transcriptions à chaque évaluation.
`remove_unused_columns=False` empêche le `Trainer` de jeter `input_features`.
La détection de checkpoint permet de **reprendre** un run interrompu (Kaggle en arrière-plan).

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size = 8,
    gradient_accumulation_steps = 2,        # batch effectif = 16
    per_device_eval_batch_size  = 8,
    learning_rate = 1e-5,
    warmup_steps  = 200,
    max_steps     = 4000,                   # ~ ajuster selon le temps dispo
    gradient_checkpointing = True,
    fp16 = (device == "cuda"),
    predict_with_generate = True,
    generation_max_length = MAX_LABEL_LEN,
    eval_strategy = "steps",
    eval_steps    = 500,
    save_steps    = 500,
    logging_steps = 25,
    load_best_model_at_end = True,
    metric_for_best_model  = "wer",
    greater_is_better      = False,         # WER : plus bas = mieux
    save_total_limit       = 2,
    remove_unused_columns  = False,         # garder input_features
    disable_tqdm           = True,
    report_to              = "none",
)

trainer = Seq2SeqTrainer(
    args             = training_args,
    model            = model,
    train_dataset    = ds_prep["train"],
    eval_dataset     = ds_prep["validation"],
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
    processing_class = processor,
)
print("Trainer Whisper pret.")


In [ ]:
last_ckpt = None
output_path = Path(OUTPUT_DIR)
if output_path.is_dir():
    ckpts = sorted(
        [d for d in output_path.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]),
    )
    if ckpts:
        last_ckpt = str(ckpts[-1])
        print(f"Reprise depuis : {last_ckpt}")

train_result = trainer.train(resume_from_checkpoint=last_ckpt)

trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"\nModele Whisper sauvegarde : {OUTPUT_DIR}")
print(f"Loss train finale : {train_result.training_loss:.4f}")


## 9. Évaluation finale + transcription d'un exemple

On mesure le WER/CER sur le **test set**, puis on transcrit un fichier audio réel pour voir
le résultat « en vrai ».

In [ ]:
metrics = trainer.evaluate(ds_prep["test"])
print(f"WER test : {metrics['eval_wer']:.2f} %")
print(f"CER test : {metrics['eval_cer']:.2f} %")


In [ ]:
# Transcrire un exemple audio brut du test set
model.eval()

def transcrire(audio_array, sr=SAMPLING_RATE):
    feats = processor(audio_array, sampling_rate=sr, return_tensors="pt").input_features.to(device)
    with torch.no_grad():
        ids = model.generate(feats, max_new_tokens=MAX_LABEL_LEN)
    return processor.batch_decode(ids, skip_special_tokens=True)[0]

ex = ds["test"][0]
pred = transcrire(ex["audio"]["array"], ex["audio"]["sampling_rate"])
print("Reference :", ex["transcription"])
print("Predit    :", pred)


## 10. Pour aller plus loin

- **Publier le modèle :** `model.push_to_hub("votre-nom/whisper-ewe")` et
  `processor.push_to_hub(...)`.
- **Alternative native éwé — MMS (Meta) :** `facebook/mms-1b-all` prend en charge l'éwé
  *nativement* (>1000 langues). C'est un modèle **CTC** (wav2vec2), pipeline un peu
  différent : on renomme `transcription → sentence`, on construit un vocabulaire de
  caractères, et on entraîne avec une perte CTC. Souvent **meilleur** que Whisper pour les
  langues africaines peu dotées.
- **Réutilisation :** la sortie texte de ce modèle ASR peut alimenter directement le
  notebook de **traduction** (éwé → français/anglais), puis le notebook **TTS** pour
  reparler — c'est le pipeline complet *voix → voix*.
